# 02. 전처리와 SQL 적재

- **목적**: 커피-음료만 남기고 분석에 쓸 컬럼만 골라 PostgreSQL에 표 3개로 적재
- **데이터 출처**: `data/raw/`의 원자료 11개 (`01`과 같음)
- **출력**: DB(`cafe_location_project`) 표 `커피_매출`, `커피_점포`, `영역`

## 설정

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

sys.path.append("..")  # notebooks/에서 실행하므로 프로젝트 루트의 src를 import 경로에 추가
from src.data_io import CODE_DTYPES, load_yearly_csv
from src.db import get_engine

In [2]:
RAW_DIR = Path("../data/raw")
ENV_PATH = Path("../.env")
PK_SQL_PATH = Path("../sql/01_기본키.sql")
AREA_FILE = "서울시_상권분석서비스(영역-상권).csv"

CAFE_CODE = "CS100010"  # 커피-음료
STORE_RENAME = {"점포_수": "일반_점포_수", "유사_업종_점포_수": "전체_점포_수"}  # 2025년 파일부터 새 이름 사용

# 분석에 쓰는 컬럼만 남김. 상권 이름은 영역 표에만 두고 필요할 때 JOIN함
SALES_COLS = ["기준_년분기_코드", "상권_코드", "당월_매출_금액", "당월_매출_건수"]
STORE_COLS = ["기준_년분기_코드", "상권_코드", "전체_점포_수"]
AREA_COLS = ["상권_코드", "상권_코드_명", "상권_구분_코드_명", "자치구_코드_명"]

## 데이터 로드

In [3]:
raw_sales = load_yearly_csv(RAW_DIR, "*추정매출*.csv")
raw_store = load_yearly_csv(RAW_DIR, "*점포-상권*.csv", rename_cols=STORE_RENAME)
raw_area = pd.read_csv(RAW_DIR / AREA_FILE, encoding="cp949", dtype=CODE_DTYPES)

서울시_상권분석서비스(추정매출-상권)_2021년.csv (89150, 55)


서울시_상권분석서비스(추정매출-상권)_2022년.csv (88834, 55)


서울시_상권분석서비스(추정매출-상권)_2023년.csv (88246, 55)


서울시_상권분석서비스(추정매출-상권)_2024년.csv (87179, 55)


서울시_상권분석서비스(추정매출-상권)_2025~2026년.csv (128053, 55)


서울시_상권분석서비스(점포-상권)_2021년.csv (303880, 14)


서울시_상권분석서비스(점포-상권)_2022년.csv (305587, 14)


서울시_상권분석서비스(점포-상권)_2023년.csv (307741, 14)


서울시_상권분석서비스(점포-상권)_2024년.csv (306889, 14)


서울시_상권분석서비스(점포-상권)_2025~2026년.csv (456659, 14)


## 전처리

| 문제 (`01`에서 확인) | 처리 |
|---|---|
| 다른 업종도 섞여 있음 | 커피-음료(`CS100010`)만 남김 |
| 컬럼이 너무 많음(매출 55개) | 분석에 쓰는 컬럼만 남김 |
| 상권 이름의 가운뎃점(·)이 `?`로 깨짐 | `?`를 `·`로 바꿈 |
| 상권 코드 하나에 이름이 2개 | 매출·점포 표에서는 이름을 빼고, 이름은 영역 표 하나에만 둠 |

In [4]:
cafe_sales = raw_sales.loc[raw_sales["서비스_업종_코드"] == CAFE_CODE, SALES_COLS]
cafe_store = raw_store.loc[raw_store["서비스_업종_코드"] == CAFE_CODE, STORE_COLS]
area = raw_area[AREA_COLS].assign(상권_코드_명=lambda d: d["상권_코드_명"].str.replace("?", "·", regex=False))
print("커피 매출", len(cafe_sales), "행 / 커피 점포", len(cafe_store), "행 / 영역", len(area), "행")

커피 매출 23700 행 / 커피 점포 33263 행 / 영역 1650 행


## 검증

In [5]:
checks = {
    "매출 키 중복": cafe_sales.duplicated(["기준_년분기_코드", "상권_코드"]).sum(),
    "점포 키 중복": cafe_store.duplicated(["기준_년분기_코드", "상권_코드"]).sum(),
    "영역에 없는 상권(매출)": (~cafe_sales["상권_코드"].isin(area["상권_코드"])).sum(),
    "이름에 '?' 남음": area["상권_코드_명"].str.contains("?", regex=False).sum(),
    "빈 값": cafe_sales.isna().sum().sum() + cafe_store.isna().sum().sum() + area.isna().sum().sum(),
}
assert all(v == 0 for v in checks.values()), checks
checks

{'매출 키 중복': np.int64(0),
 '점포 키 중복': np.int64(0),
 '영역에 없는 상권(매출)': np.int64(0),
 "이름에 '?' 남음": np.int64(0),
 '빈 값': np.int64(0)}

## SQL 적재

- `if_exists="replace"`로 매번 표를 새로 만듦 → 노트북을 다시 실행해도 결과가 같음
- 적재한 뒤 DB 행 수를 DataFrame 행 수와 비교

In [6]:
engine = get_engine(ENV_PATH)

tables = {"커피_매출": cafe_sales, "커피_점포": cafe_store, "영역": area}
for name, df in tables.items():
    df.to_sql(name, engine, if_exists="replace", index=False)
    with engine.connect() as conn:
        n_loaded = conn.execute(text(f'SELECT COUNT(*) FROM "{name}"')).scalar()
    print(name, "| DataFrame", len(df), "| DB", n_loaded)
    assert len(df) == n_loaded

커피_매출 | DataFrame 23700 | DB 23700


커피_점포 | DataFrame 33263 | DB 33263
영역 | DataFrame 1650 | DB 1650


## 기본 키 설정

- 기본 키는 같은 (분기, 상권) 행이 두 번 들어가는 것을 DB가 막게 하는 장치
- `replace`로 표를 다시 만들면 기본 키가 사라지므로 적재할 때마다 다시 실행

In [7]:
statements = [s for s in PK_SQL_PATH.read_text(encoding="utf-8").split(";") if s.strip()]
with engine.begin() as conn:  # 하나라도 실패하면 모두 취소
    for statement in statements:
        conn.execute(text(statement))
print("기본 키", len(statements), "개 설정")

기본 키 3 개 설정


## 결론과 다음 단계

- 커피 매출 23,700행, 커피 점포 33,263행, 영역 1,650행을 DB에 적재, DataFrame과 DB 행 수 일치
- 검증 항목 5개 모두 0건, 기본 키 3개 설정
- 다음 단계: `03_추천.ipynb`에서 `sql/02_추천지표.sql`로 지표를 계산하고 후보를 고름